# Model M3: Expenditure & Payment Behavioral Screening (Google Colab Notebook)
**SIH26102 MPLADS Audit Intelligence Platform**

Tests Model M3 (Transaction-Grain Payment Structuring & Velocity Screening).

In [ ]:
# Self-contained Repository & Path Setup for Google Colab
import os, sys, glob, zipfile

# 1. Look for unified_model_engine.py in current dir or subdirs
if not os.path.exists("unified_model_engine.py"):
    subdirs = [d for d in os.listdir(".") if os.path.isdir(d) and os.path.exists(os.path.join(d, "unified_model_engine.py"))]
    if subdirs:
        os.chdir(subdirs[0])
    else:
        zip_files = glob.glob("*.zip") + glob.glob("../*.zip")
        if zip_files:
            print(f"Extracting uploaded archive: {zip_files[0]}...")
            with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
                zip_ref.extractall(".")
            subdirs = [d for d in os.listdir(".") if os.path.isdir(d) and os.path.exists(os.path.join(d, "unified_model_engine.py"))]
            if subdirs:
                os.chdir(subdirs[0])
        else:
            !git clone https://github.com/Zaid5671/CodeBlooded.git
            if os.path.exists("CodeBlooded"):
                os.chdir("CodeBlooded")

sys.path.insert(0, os.getcwd())
os.environ["PYTHONPATH"] = os.getcwd()
print("Working Directory:", os.getcwd())
print("Engine Available:", os.path.exists("unified_model_engine.py"))
!pip install -q pandas numpy scikit-learn pytest

In [ ]:
# Run Model M3 Expenditure Behavioral Screening Directly
import pandas as pd
from data_pipeline.data_loader import load_sanctioned_works, load_expenditure_works
from feature_engineering.preprocessing import preprocess_sanctioned_works
from feature_engineering.category_classifier import apply_category_classification
from ml.model_3_expenditure_anomaly.expenditure_matching import match_expenditure_data
from audit_rules.statutory_compliance.compliance_rules import evaluate_compliance_rules
from audit_rules.vendor_risk.vendor_analytics import evaluate_vendor_analytics

print("=== TESTING MODEL M3 EXPENDITURE SCREENING ENGINE ===")
df_sanc = load_sanctioned_works()
df_exp = load_expenditure_works()
df_prep = preprocess_sanctioned_works(df_sanc)
df_cat = apply_category_classification(df_prep)
df_m3 = match_expenditure_data(df_cat, df_exp)

matched_count = df_m3['actual_expenditure'].notnull().sum()
print(f"Total Raw Expenditure Transactions: {len(df_exp):,}")
print(f"Sanctioned Works with Matched Expenditure: {matched_count:,} / {len(df_m3):,}")
print(f"Single-Payment Works (1 voucher): {(df_m3['num_payments'] == 1).sum():,}")
print(f"Multi-Payment Works (>1 vouchers): {(df_m3['num_payments'] > 1).sum():,}")

# Display Top 5 Multi-Payment Works
df_m3[df_m3['num_payments'] > 1][['clean_work_id', 'actual_expenditure', 'num_payments', 'max_payment_ratio']].head()